# Comparación de modelos en RWF-2000

Este notebook compara los resultados **medidos** en un subset de 400 clips: 259 de entrenamiento, 41 de validación interna y 100 de test tomados de la carpeta `val/` original. Se ejecutó en CPU con semilla 42, batch 4, tasa de aprendizaje 1e-4 y hasta 5 épocas. El LSTM se detuvo tras 4 épocas por parada temprana. **No representa una evaluación del dataset completo de 2.000 clips.**

Clase positiva: `Fight` (Sospechoso). Las métricas proceden de `artifacts/metrics_<modelo>.json`, nunca de valores escritos a mano.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
# Se puede ejecutar desde la raíz del repositorio o desde notebooks/.
root = Path.cwd()
if not (root / "artifacts").is_dir():
    root = root.parent
artifacts_dir = root / "artifacts"
model_names = ["mobilenet_lstm", "mobilenet_baseline", "resnet_lstm"]
reports = {}
for model_name in model_names:
    metrics_path = artifacts_dir / f"metrics_{model_name}.json"
    if metrics_path.is_file():
        reports[model_name] = json.loads(metrics_path.read_text(encoding="utf-8"))
    else:
        print(f"TODO: falta {metrics_path.name}; entrenar/evaluar {model_name} y volver a ejecutar.")
if not reports:
    raise FileNotFoundError(f"No se encontraron métricas en {artifacts_dir.resolve()}")

In [ ]:
# La fila pendiente permanece visible sin atribuirle resultados inventados.
rows = []
for model_name in model_names:
    report = reports.get(model_name)
    if report is None:
        rows.append({"Modelo": model_name, "Estado": "PENDIENTE"})
        continue
    final = report["final"]
    positive = report["class_names"][1]
    rows.append({
        "Modelo": model_name,
        "Estado": "Medido",
        "Accuracy": final["accuracy"],
        "F1 macro": final["f1_macro"],
        "Recall Sospechoso": final["recall_per_class"][positive],
        "Precisión Sospechoso": final["precision_per_class"][positive],
        "AP": report["pr_curve"]["average_precision"] if report.get("pr_curve") else np.nan,
        "ms/clip": report.get("inference_ms", np.nan),
        "Matriz [[TN, FP], [FN, TP]]": str(final["confusion_matrix"]),
    })
comparison = pd.DataFrame(rows)
display(comparison.round(3))

## Lectura de la tabla

Cada fila medida corresponde al **mismo test de 100 clips** y a la clase positiva `Fight`. Accuracy es la proporción de aciertos; F1 macro promedia el F1 de ambas clases; AP resume la curva precisión–recall. La matriz sigue el orden real/predicho `[NonFight, Fight]`. La latencia es por clip en CPU y depende del equipo. `resnet_lstm` queda **PENDIENTE** hasta que exista su JSON de test.

In [ ]:
available = [name for name in model_names if name in reports]
fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 4), squeeze=False)
for ax, name in zip(axes[0], available):
    matrix = np.asarray(reports[name]["final"]["confusion_matrix"])
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["Normal", "Sospechoso"],
                yticklabels=["Normal", "Sospechoso"], ax=ax)
    ax.set(title=name, xlabel="Predicción", ylabel="Real")
fig.suptitle("Matrices de confusión · test de 100 clips")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name in available:
    curve = reports[name].get("pr_curve")
    if not curve:
        print(f"TODO: falta la curva PR de {name}.")
        continue
    ax.plot(curve["recall"], curve["precision"],
            label=f'{name} (AP={curve["average_precision"]:.3f})')
ax.set(xlabel="Recall Sospechoso", ylabel="Precisión Sospechoso",
       title="Curvas precisión–recall · test", xlim=(0, 1), ylim=(0, 1))
ax.grid(alpha=0.25)
ax.legend()
plt.show()

In [ ]:
measured = comparison[comparison["Estado"] == "Medido"]
positions = np.arange(len(measured))
width = 0.36
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(positions - width / 2, measured["Recall Sospechoso"], width, label="Recall")
ax.bar(positions + width / 2, measured["Precisión Sospechoso"], width, label="Precisión")
ax.set_xticks(positions, measured["Modelo"])
ax.set_ylim(0, 1)
ax.set_ylabel("Puntuación")
ax.set_title("Clase Sospechoso · test")
ax.legend()
fig.tight_layout()
plt.show()

## Conclusión y decisión

- **`mobilenet_baseline`** obtuvo mayor accuracy (0,660 frente a 0,640), F1 macro (0,659 frente a 0,638), precisión de Sospechoso (0,674 frente a 0,621), AP (0,684 frente a 0,651) y menor latencia (406 frente a 594 ms/clip, redondeado).
- **`mobilenet_lstm`** obtuvo mayor recall de Sospechoso (0,720 frente a 0,620): dejó sin detectar 14 peleas frente a 19. A cambio, generó 22 falsas alarmas frente a 15.
- Para la **demo académica** se elige `mobilenet_lstm`, porque priorizamos reducir falsos negativos de `Fight`. Esta decisión refleja una preferencia de costo entre errores; no demuestra seguridad operativa ni detección de asaltos, una categoría que RWF-2000 no etiqueta.
- El test tiene solo 100 clips. La diferencia de F1 macro es aproximadamente **2,2 puntos porcentuales** y puede variar con otra muestra. El umbral, la duración de la alerta y la generalización a cámaras bolivianas necesitan validación adicional. Estudios sobre RWF-2000 reportan resultados mayores con otros métodos, pero **no permiten prometer** un F1 de 0,78–0,85 para este modelo.
- **TODO:** cuando se genere `artifacts/metrics_resnet_lstm.json`, volver a ejecutar todas las celdas y revisar la decisión.